In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import sklearn.metrics.pairwise as dist

Chargement des données

In [16]:
ratings = pd.read_csv('ratings_cleaned.csv')
movies = pd.read_csv('movies_cleaned.csv')

MOVIES

In [17]:
display(movies.head(2))
movies.shape

,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(27278, 21)

RATINGS

In [18]:
display(ratings.head(2))
ratings.shape

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5


(20000263, 3)

**********************************************************************

Filtrage collaboratif : on exploite exclusivement les interactions passées dans les utilisateurs et les films, en regroupant et identifiant des groupes d'utilisateurs dont les interactions sont similaires

**********************************************************************

1. Approche mémoire : elle se base sur la corrélation entre les comportements "passés" des utilisateurs

In [ ]:
# on crée un dataframe df qui va contenir ratings mais avec les titres de film en plus des identifiants
# On ne s'intéresse qu'aux notes >= 3, car en dessous les notes sont moins pertinentes
df = ratings[ratings['rating'] >= 3]

n_users = df['userId'].nunique()
n_movies = df['movieId'].nunique()
print("Nombre d'utilisateurs : ", n_users)
print("Nombre de films : ", n_movies)
print(f"La matrice dense aura pour taille : {n_users*n_movies:,}")

Nombre d'utilisateurs :  138445
Nombre de films :  24800
La matrice dense aura pour taille : 3,433,436,000


Nous avons 2 types d'approche mémoire : User-based filtering (similarité entre les utilisateurs) et Item-based filtering (similarité entre les films). La similarité sur les utilisateurs va construire une matrice de dimension 138445*138445 soit 19 167 018 025 cellules. Cette méthode n'est pas envisageable sur un tel nombre d'utilisateurs. Il faudrait réduire considérablement le nombre d'utilisateurs ce qui biaiserait énormément notre jeu de données. En alternative on pourrait mettre en place la méthode des k plus proches voisins. Nous n'appliquerons donc que la méthode basée sur la similarité entre les films

Item-based filtering : on ne mesure pas la corrélation entre des utilisateurs mais entre le contenu (films). Le but est de trouver des films similaire aux films que l'utilisateur cible a beaucoup aimés. La matrice de similarité aura pour dimension 24800 * 24800, soit 615 040 000 cellules. Le calcul est envisageable, mais il va devoir passer par une matrice de notation de dimension 24800 * 138445, ce qui est assez coûteux.

In [20]:
# matrice mat_ratings de notation associée au dataframe en prenant en index les movieId et en colonnes les userId
mat_ratings = df.pivot(index='movieId', columns='userId', values='rating')
display(mat_ratings.info())
print("Dimension de la matrice de notation : ", mat_ratings.shape)
mat_ratings.head(2)

/var/folders/43/j4zwb6d13hd0m383lzvlyljr0000gn/T/ipykernel_87472/381608062.py:2: PerformanceWarning: The following operation may generate 3433436000 cells in the resulting pandas object.
  mat_ratings = df.pivot(index='movieId', columns='userId', values='rating')


<class 'pandas.DataFrame'>
Index: 24800 entries, 1 to 131262
Columns: 138445 entries, 1 to 138493
dtypes: float64(138445)
memory usage: 25.6 GB


None

Dimension de la matrice de notation :  (24800, 138445)


userId,1,2,3,4,5,6,7,8,9,10,...,138484,138485,138486,138487,138488,138489,138490,138491,138492,138493
movieId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,4.0,NaN,NaN,5.0,NaN,4.0,NaN,4.0,...,NaN,NaN,5.0,NaN,3.0,NaN,NaN,NaN,NaN,3.5
2,3.5,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,3.0,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,4.0


La matrice de notation est assez dense et fait 25,6GB ce qui est assez conséquent pour la mémoire de nos ordinateurs

Alternative : En construisant la matrice creuse **directement à partir des triplets** (movieId, userId, rating), sans jamais matérialiser de tableau dense, on peut optimiser le temps de calcul 

In [21]:
# encodage des movieId et userId en indices entiers consécutifs (0..n-1), nécessaire pour
# construire directement la matrice creuse à partir des triplets
movie_cat = df['movieId'].astype('category')
user_cat = df['userId'].astype('category')

moviesId = movie_cat.cat.categories.tolist()
userIds = user_cat.cat.categories.tolist()

# construction de la matrice creuse (films x users) sans passer par une matrice dense
sparse_ratings = csr_matrix(
    (df['rating'].values, (movie_cat.cat.codes.values, user_cat.cat.codes.values)),
    shape=(len(moviesId), len(userIds))
)

print("Dimension de la matrice creuse :", sparse_ratings.shape)
print(sparse_ratings)

Dimension de la matrice creuse : (24800, 138445)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16486759 stored elements and shape (24800, 138445)>
  Coords	Values
  (0, 2)	4.0
  (0, 5)	5.0
  (0, 7)	4.0
  (0, 9)	4.0
  (0, 10)	4.5
  (0, 11)	4.0
  (0, 12)	4.0
  (0, 13)	4.5
  (0, 15)	3.0
  (0, 18)	5.0
  (0, 21)	3.0
  (0, 22)	4.0
  (0, 23)	4.0
  (0, 30)	3.0
  (0, 33)	5.0
  (0, 38)	5.0
  (0, 52)	4.0
  (0, 53)	4.0
  (0, 57)	5.0
  (0, 58)	4.5
  (0, 65)	4.0
  (0, 68)	4.0
  (0, 79)	3.0
  (0, 81)	5.0
  (0, 83)	5.0
  :	:
  (24776, 79539)	4.0
  (24777, 79539)	4.0
  (24778, 79539)	3.5
  (24779, 79539)	4.0
  (24780, 79539)	4.0
  (24780, 138019)	4.0
  (24781, 79539)	4.0
  (24782, 54542)	4.0
  (24783, 54542)	4.0
  (24784, 64039)	3.5
  (24785, 95803)	3.5
  (24786, 109243)	3.5
  (24787, 109243)	4.5
  (24788, 63025)	3.5
  (24789, 134653)	3.0
  (24790, 79539)	4.0
  (24791, 79539)	4.0
  (24792, 79539)	4.0
  (24793, 79539)	4.0
  (24794, 79539)	4.0
  (24795, 79539)	4.0
  (24796, 79539)	4.0
  (

In [22]:
# Utilisation de la fonction 'cosine_similarity' du module 'dist' pour calculer la similarité cosinus entre les utilisateurs.
item_similarity = dist.cosine_similarity(sparse_ratings)
# Création d'un DataFrame pandas à partir de la matrice de similarité entre items.
# Les index et les colonnes du DataFrame sont les identifiants des films.
item_similarity = pd.DataFrame(item_similarity, index=moviesId, columns=moviesId)
# on nomme l'index pour que reset_index() produise une colonne 'movieId' (nécessaire pour les merges)
item_similarity.index.name = 'movieId'

In [23]:
# fonction qui prend en entrée un movieId et renvoie les 10 films les plus similaires à ce film
def get_similar_movies(movieId, item_similarity, top_n=10):
    # Vérifie si le movieId est présent dans la matrice de similarité
    if movieId not in item_similarity.index:
        return pd.DataFrame()  # Retourne un DataFrame vide si le movieId n'est pas trouvé

    # Récupère les similarités pour le film donné
    similar_scores = item_similarity[movieId]

    # Trie les films par similarité décroissante et sélectionne les top_n films similaires
    top_similar_movies = similar_scores.sort_values(ascending=False).head(top_n + 1)  # +1 pour exclure le film lui-même

    # Exclut le film lui-même de la liste des films similaires
    top_similar_movies = top_similar_movies[top_similar_movies.index != movieId]

    return top_similar_movies

In [24]:
#recherche de film à tester
movies[movies['title'].str.contains('wars', case=False)]

,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
257,260,Star Wars: Episode IV - A New Hope (1977),1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1171,1196,Star Wars: Episode V - The Empire Strikes Back...,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1184,1210,Star Wars: Episode VI - Return of the Jedi (1983),1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2543,2628,Star Wars: Episode I - The Phantom Menace (1999),1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
5281,5378,Star Wars: Episode II - Attack of the Clones (...,1,1,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
5345,5442,V. I. Warshawski (1991),1,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
8047,8730,To End All Wars (2001),1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
9184,27067,"Pentagon Wars, The (1998)",0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
9223,27176,Style Wars (1983),0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
9549,27884,Word Wars (2004),0,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [25]:
# recherche l'utlisateur qui a regardé le movieId 1196
mat_ratings.loc[1196].dropna()

userId
1         4.5
2         5.0
3         5.0
5         5.0
7         5.0
         ... 
138472    4.0
138474    5.0
138477    5.0
138492    4.5
138493    4.5
Name: 1196, Length: 43003, dtype: float64

In [26]:
movieId = 89745
similar_movies = get_similar_movies(movieId, item_similarity, top_n=20)
# Ajouter les titres des films similaires
similar_movies = similar_movies.reset_index()
similar_movies = similar_movies.merge(movies[['movieId', 'title']], left_on='movieId', right_on='movieId', how='left')
print(similar_movies)

    movieId     89745                                              title
0     91529  0.603283                      Dark Knight Rises, The (2012)
1     87232  0.556043                          X-Men: First Class (2011)
2     88140  0.524616          Captain America: The First Avenger (2011)
3     77561  0.512900                                  Iron Man 2 (2010)
4     86332  0.512238                                        Thor (2011)
5    102125  0.509441                                  Iron Man 3 (2013)
6     91500  0.497690                           Hunger Games, The (2012)
7     95510  0.492810                     Amazing Spider-Man, The (2012)
8     96079  0.471160                                     Skyfall (2012)
9     98809  0.468161          Hobbit: An Unexpected Journey, The (2012)
10    91542  0.467713          Sherlock Holmes: A Game of Shadows (2011)
11    96610  0.457966                                      Looper (2012)
12    88125  0.453682  Harry Potter and the Deathly

In [27]:
def pred_item(mat_ratings, item_similarity, k, user_id):

    # Sélectionner dans mat_ratings les films qui n'ont pas été encore vus par le user
    # (mat_ratings a movieId en index et userId en colonnes, donc on sélectionne la colonne user_id)
    to_predict = mat_ratings[user_id]
    to_predict = to_predict[to_predict.isna()]
    print(f"Nombre de films à prédire pour l'utilisateur {user_id} : {len(to_predict)}")

    # Itérer sur tous ces films 
    for i in to_predict.index:

        #Trouver les k films les plus similaires en excluant le film lui-même
        similar_items = item_similarity.loc[i].sort_values(ascending=False)[1:k+1]

        # Calcul de la norme du vecteur similar_items
        norm = np.sum(np.abs(similar_items))

        # Récupérer les notes données par l'utilisateur aux k plus proches voisins
        ratings = mat_ratings.loc[similar_items.index, user_id].fillna(0)


        # Calculer le produit scalaire entre ratings et similar_items
        scalar_prod = np.dot(ratings,similar_items)
        
        #Calculer la note prédite pour le film i
        pred = scalar_prod / norm

        # Remplacer par la prédiction
        to_predict[i] = pred


    return to_predict

In [28]:
# Top notations de l'utilisateur userId
#userId = 31
userId = 1
user_preferences = df[(df['userId']==userId) & (df['rating']>=4)]
user_preferences.sort_values('rating', ascending=False).drop_duplicates().head(30)

,userId,movieId,rating
170,1,8507,5.0
142,1,5952,5.0
158,1,7153,5.0
131,1,4993,5.0
30,1,1196,4.5
171,1,8636,4.5
31,1,1198,4.5
113,1,4011,4.0
126,1,4896,4.0
124,1,4754,4.0


In [29]:
reco_item = pred_item(mat_ratings, item_similarity, 10, userId).sort_values(ascending=False).head(30)

# ajouter les titres des films recommandés
reco_item = reco_item.reset_index()
reco_item = reco_item.merge(movies[['movieId', 'title']], left_on='movieId', right_on='movieId', how='left')

print(reco_item)

Nombre de films à prédire pour l'utilisateur 1 : 24625
    movieId         1                                              title
0      5782  3.800749        Professional, The (Le professionnel) (1981)
1      1275  3.257785                                  Highlander (1986)
2      2571  3.240632                                 Matrix, The (1999)
3      1242  3.215725                                       Glory (1989)
4      1220  3.156885                         Blues Brothers, The (1980)
5      5349  3.153883                                  Spider-Man (2002)
6      6874  3.083269                           Kill Bill: Vol. 1 (2003)
7      2467  3.069488  Name of the Rose, The (Name der Rose, Der) (1986)
8      1345  3.033460                                      Carrie (1976)
9      1343  3.004783                                   Cape Fear (1991)
10     1206  2.968941                         Clockwork Orange, A (1971)
11     7361  2.955371       Eternal Sunshine of the Spotless Mind (20